# Imports


In [33]:
from __future__ import annotations

import math
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

# Constants


In [34]:
RANDOM_SEED = 42

In [35]:
DATA_DIR: Path = Path("/home/linkezio/Datasets/cifar-10-python/cifar-10-batches-py")

In [36]:
MODELS_DIR: Path = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks/models")

# Configs


## Seeds


In [37]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Device


In [38]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


# Data


## Dataset Class


In [39]:
# Loads one CIFAR batch and returns images and labels.
def _load_cifar_batch(path: Path) -> tuple[np.ndarray, np.ndarray]:
    with path.open("rb") as f:
        obj = pickle.load(f, encoding="bytes")
    data = obj[b"data"]  # (N, 3072)
    labels = np.array(obj.get(b"labels") or obj.get(b"fine_labels"), dtype=np.int64)
    images = data.reshape(-1, 3, 32, 32)
    return images, labels


In [40]:
class CIFAR10Dataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        train: bool,
        mean: torch.Tensor | None = None,
        std: torch.Tensor | None = None,
    ):
        if train:
            batch_files = [data_dir / f"data_batch_{i}" for i in range(1, 6)]
        else:
            batch_files = [data_dir / "test_batch"]

        xs = []
        ys = []
        for p in batch_files:
            if not p.exists():
                raise FileNotFoundError(f"File not found: {p}")
            x, y = _load_cifar_batch(p)
            xs.append(x)
            ys.append(y)

        images = np.concatenate(xs, axis=0)
        labels = np.concatenate(ys, axis=0)

        self.images = torch.from_numpy(images).float()  # (N,3,32,32)
        self.labels = torch.from_numpy(labels).long()
        
        if (mean is None) ^ (std is None):
            raise ValueError("Pass `mean` and `std` together, or neither (data in [0, 1]).")
        self.mean = mean
        self.std = std

    # Returns the total number of dataset samples.
    def __len__(self) -> int:
        return int(self.labels.shape[0])

    # Returns one sample (x, y), with optional normalization.
    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.mean is not None:
            x = (x - self.mean) / self.std
        y = self.labels[idx]
        return x, y


## Calculate mean and std for normalize later


In [41]:

# Computes per-channel mean and standard deviation over the full dataset.
def compute_mean_std(dataset, batch_size=512):
    loader_mean = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    channel_sum = None
    n_pixels = 0
    for images, _ in loader_mean:
        b, c, h, w = images.shape
        if channel_sum is None:
            channel_sum = torch.zeros(c, dtype=torch.float64)
        channel_sum += images.double().sum(dim=(0, 2, 3))
        n_pixels += b * h * w

    mean = (channel_sum / n_pixels).float()

    loader_var = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sum_sq = None
    for images, _ in loader_var:
        b, c, h, w = images.shape
        if sum_sq is None:
            sum_sq = torch.zeros(c, dtype=torch.float64)
        diff = images.double() - mean.view(1, c, 1, 1).double()
        sum_sq += (diff * diff).sum(dim=(0, 2, 3))

    var = (sum_sq / n_pixels).float()
    std = torch.sqrt(var)
    std = torch.clamp(std, min=1e-8)
    return mean, std


In [42]:
train_for_stats = CIFAR10Dataset(DATA_DIR, train=True)

cifar_mean, cifar_std = compute_mean_std(train_for_stats)

cifar_mean = cifar_mean.view(3, 1, 1)
cifar_std = cifar_std.view(3, 1, 1)

print("mean (R,G,B):", cifar_mean.squeeze().tolist())
print("std  (R,G,B):", cifar_std.squeeze().tolist())

/tmp/ipykernel_113132/3686208571.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f, encoding="bytes")


mean (R,G,B): [125.30691528320312, 122.95039367675781, 113.86538696289062]
std  (R,G,B): [62.993221282958984, 62.088706970214844, 66.70490264892578]


## Train, Validation, Test Split


In [43]:
class DataLoaderHyperparameters:
    batch_size: int = 128
    val_fraction: float = 0.1
    num_workers: int = 0 # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).

data_loader_hyperparameters = DataLoaderHyperparameters()

In [44]:
full_train = CIFAR10Dataset(DATA_DIR, train=True, mean=cifar_mean, std=cifar_std)
test_ds = CIFAR10Dataset(DATA_DIR, train=False, mean=cifar_mean, std=cifar_std)

val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=True,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

len(train_ds), len(val_ds), len(test_ds)


/tmp/ipykernel_113132/3686208571.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f, encoding="bytes")


(45000, 5000, 10000)

# Model


## Hyperparameters


In [45]:
class ModelHyperparameters:
    batch_size: int = 128
    epochs: int = 25
    lr: float = 1e-2
    weight_decay: float = 5e-4
    # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).
    num_workers: int = 0

model_hyperparameters = ModelHyperparameters()

## Model Class


In [46]:
class SimpleCIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 10),
        )

    # Runs the model forward pass.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Train


## Metric Functions


In [47]:
# Computes average batch accuracy.
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


In [48]:
loss_fn = nn.CrossEntropyLoss()

## Epoch Functions


In [49]:
@torch.inference_mode()
# Evaluates the model for one epoch and returns average loss/accuracy.
def eval_epoch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        accs.append(accuracy(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


# Trains the model for one epoch and returns average loss/accuracy.
def train_epoch(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        accs.append(accuracy(logits.detach(), y))
    return float(np.mean(losses)), float(np.mean(accs))


## Trainings

### Training X (Baseline)


In [50]:
# Coordinates epoch training/validation and saves the best checkpoint.
def fit_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    """Coordinates epoch training/validation and saves the best checkpoint."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(float(optim.param_groups[0]["lr"]))

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [51]:
model = SimpleCIFAR10CNN().to(DEVICE)
optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [52]:
history, best_val_acc, model_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "cifar10_best.pt",
)

epoch 01/25 | train loss 2.0768 acc 0.1980 | val loss 1.8310 acc 0.2748
epoch 02/25 | train loss 1.7728 acc 0.3069 | val loss 1.7161 acc 0.3242
epoch 03/25 | train loss 1.6814 acc 0.3532 | val loss 1.6183 acc 0.3787
epoch 04/25 | train loss 1.6167 acc 0.3826 | val loss 1.5916 acc 0.4020
epoch 05/25 | train loss 1.5746 acc 0.4038 | val loss 1.5676 acc 0.4203
epoch 06/25 | train loss 1.5514 acc 0.4207 | val loss 1.5435 acc 0.4340
epoch 07/25 | train loss 1.4987 acc 0.4450 | val loss 1.4912 acc 0.4637
epoch 08/25 | train loss 1.4682 acc 0.4584 | val loss 1.4751 acc 0.4648
epoch 09/25 | train loss 1.4267 acc 0.4759 | val loss 1.4011 acc 0.4764
epoch 10/25 | train loss 1.4002 acc 0.4860 | val loss 1.3809 acc 0.5064
epoch 11/25 | train loss 1.3722 acc 0.4997 | val loss 1.4099 acc 0.4877
epoch 12/25 | train loss 1.4382 acc 0.4786 | val loss 1.3619 acc 0.5018
epoch 13/25 | train loss 1.3524 acc 0.5080 | val loss 1.3752 acc 0.5012
epoch 14/25 | train loss 1.3315 acc 0.5183 | val loss 1.4580 acc

### Training Y (Library LR Scheduler)


In [53]:
class SchedulerHyperparameters:
    epochs: int = model_hyperparameters.epochs
    max_lr: float = model_hyperparameters.lr
    min_lr: float = 0.00000000000000000000000000000000001

scheduler_hyperparameters = SchedulerHyperparameters()


In [54]:
# Trains with a built-in PyTorch scheduler (CosineAnnealingWarmRestarts).
def fit_model_with_library_scheduler(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    scheduler_hp: SchedulerHyperparameters,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optim,
        T_0=max(1, scheduler_hp.epochs // 4),
        T_mult=2,
        eta_min=scheduler_hp.min_lr,
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, scheduler_hp.epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        scheduler.step()
        current_lr = float(optim.param_groups[0]["lr"])

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(current_lr)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{scheduler_hp.epochs} | "
            f"lr {current_lr:.6f} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [55]:
model_scheduler = SimpleCIFAR10CNN().to(DEVICE)
optim_scheduler = torch.optim.Adam(
    model_scheduler.parameters(),
    lr=scheduler_hyperparameters.max_lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [56]:
history_scheduler, best_val_acc_scheduler, model_path_scheduler = fit_model_with_library_scheduler(
    model=model_scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim_scheduler,
    loss_fn=loss_fn,
    scheduler_hp=scheduler_hyperparameters,
    model_path=MODELS_DIR / "cifar10_best_scheduler.pt",
)


epoch 01/25 | lr 0.009330 | train loss 2.0126 acc 0.2389 | val loss 1.7226 acc 0.3293
epoch 02/25 | lr 0.007500 | train loss 1.6548 acc 0.3637 | val loss 1.5496 acc 0.4258
epoch 03/25 | lr 0.005000 | train loss 1.5022 acc 0.4368 | val loss 1.4689 acc 0.4584
epoch 04/25 | lr 0.002500 | train loss 1.3867 acc 0.4899 | val loss 1.3594 acc 0.5115
epoch 05/25 | lr 0.000670 | train loss 1.3074 acc 0.5235 | val loss 1.3089 acc 0.5254
epoch 06/25 | lr 0.010000 | train loss 1.2442 acc 0.5499 | val loss 1.2454 acc 0.5527
epoch 07/25 | lr 0.009830 | train loss 1.4088 acc 0.4893 | val loss 1.3184 acc 0.5205
epoch 08/25 | lr 0.009330 | train loss 1.3178 acc 0.5234 | val loss 1.2908 acc 0.5273
epoch 09/25 | lr 0.008536 | train loss 1.2730 acc 0.5433 | val loss 1.2809 acc 0.5359
epoch 10/25 | lr 0.007500 | train loss 1.2406 acc 0.5536 | val loss 1.2625 acc 0.5498
epoch 11/25 | lr 0.006294 | train loss 1.2132 acc 0.5631 | val loss 1.2152 acc 0.5623
epoch 12/25 | lr 0.005000 | train loss 1.1770 acc 0.57

## Training animation


In [58]:
# Side-by-side animation: baseline vs scheduler (loss curves + LR).
# Requires: run Training X and Training Y first so `history` and `history_scheduler` exist.

from IPython.display import HTML
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np


def animate_training_compare(
    history_baseline: dict,
    history_scheduler: dict,
    baseline_title: str = "Baseline (fixed LR)",
    scheduler_title: str = "CosineAnnealingWarmRestarts",
    interval_ms: int = 80,
):
    n = len(history_baseline["train_loss"])
    ns = len(history_scheduler["train_loss"])
    if n != ns:
        raise ValueError(f"Histories must have same length (got {n} vs {ns}).")

    epochs = np.arange(1, n + 1)
    tb = np.asarray(history_baseline["train_loss"], dtype=float)
    vb = np.asarray(history_baseline["val_loss"], dtype=float)
    ts = np.asarray(history_scheduler["train_loss"], dtype=float)
    vs = np.asarray(history_scheduler["val_loss"], dtype=float)

    lrb = np.asarray(history_baseline.get("lr", [float("nan")] * n), dtype=float)
    lrs = np.asarray(history_scheduler["lr"], dtype=float)

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)

    (ax_tl, ax_tr), (ax_bl, ax_br) = axes
    ax_tl.set_title(baseline_title + " — loss")
    ax_tr.set_title(scheduler_title + " — loss")
    ax_bl.set_title(baseline_title + " — learning rate")
    ax_br.set_title(scheduler_title + " — learning rate")

    for ax in (ax_tl, ax_tr):
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.grid(True, alpha=0.3)
    for ax in (ax_bl, ax_br):
        ax.set_xlabel("epoch")
        ax.set_ylabel("lr")
        ax.grid(True, alpha=0.3)

    (line_tb,) = ax_tl.plot([], [], label="train", color="C0")
    (line_vb,) = ax_tl.plot([], [], label="val", color="C1")
    dot_tb = ax_tl.scatter([], [], color="C0", s=40, zorder=5)
    dot_vb = ax_tl.scatter([], [], color="C1", s=40, zorder=5)
    ax_tl.legend(loc="upper right")

    (line_ts,) = ax_tr.plot([], [], label="train", color="C0")
    (line_vs,) = ax_tr.plot([], [], label="val", color="C1")
    dot_ts = ax_tr.scatter([], [], color="C0", s=40, zorder=5)
    dot_vs = ax_tr.scatter([], [], color="C1", s=40, zorder=5)
    ax_tr.legend(loc="upper right")

    (line_lrb,) = ax_bl.plot([], [], color="C2")
    dot_lrb = ax_bl.scatter([], [], color="C2", s=40, zorder=5)

    (line_lrs,) = ax_br.plot([], [], color="C3")
    dot_lrs = ax_br.scatter([], [], color="C3", s=40, zorder=5)

    vline_tl = ax_tl.axvline(1, color="gray", ls="--", alpha=0.5)
    vline_tr = ax_tr.axvline(1, color="gray", ls="--", alpha=0.5)
    vline_bl = ax_bl.axvline(1, color="gray", ls="--", alpha=0.5)
    vline_br = ax_br.axvline(1, color="gray", ls="--", alpha=0.5)

    def init():
        ax_tl.set_xlim(0.5, n + 0.5)
        ax_tr.set_xlim(0.5, n + 0.5)
        ax_bl.set_xlim(0.5, n + 0.5)
        ax_br.set_xlim(0.5, n + 0.5)
        y0 = float(min(tb.min(), vb.min(), ts.min(), vs.min()))
        y1 = float(max(tb.max(), vb.max(), ts.max(), vs.max()))
        pad = 0.05 * (y1 - y0 + 1e-9)
        ax_tl.set_ylim(y0 - pad, y1 + pad)
        ax_tr.set_ylim(y0 - pad, y1 + pad)
        lr_lo = float(np.nanmin([lrb.min(), lrs.min()]))
        lr_hi = float(np.nanmax([lrb.max(), lrs.max()]))
        lr_pad = 0.05 * (lr_hi - lr_lo + 1e-12)
        ax_bl.set_ylim(lr_lo - lr_pad, lr_hi + lr_pad)
        ax_br.set_ylim(lr_lo - lr_pad, lr_hi + lr_pad)
        return (
            line_tb,
            line_vb,
            line_ts,
            line_vs,
            line_lrb,
            line_lrs,
        )

    def update(k: int):
        k = int(k)
        sl = slice(0, k + 1)
        ex = epochs[sl]

        line_tb.set_data(ex, tb[sl])
        line_vb.set_data(ex, vb[sl])
        dot_tb.set_offsets(np.c_[ex[-1:], tb[sl][-1:]])
        dot_vb.set_offsets(np.c_[ex[-1:], vb[sl][-1:]])

        line_ts.set_data(ex, ts[sl])
        line_vs.set_data(ex, vs[sl])
        dot_ts.set_offsets(np.c_[ex[-1:], ts[sl][-1:]])
        dot_vs.set_offsets(np.c_[ex[-1:], vs[sl][-1:]])

        line_lrb.set_data(ex, lrb[sl])
        dot_lrb.set_offsets(np.c_[ex[-1:], lrb[sl][-1:]])

        line_lrs.set_data(ex, lrs[sl])
        dot_lrs.set_offsets(np.c_[ex[-1:], lrs[sl][-1:]])

        xcur = float(epochs[k])
        for vl in (vline_tl, vline_tr, vline_bl, vline_br):
            vl.set_xdata([xcur, xcur])

        return (
            line_tb,
            line_vb,
            line_ts,
            line_vs,
            line_lrb,
            line_lrs,
        )

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=n,
        init_func=init,
        interval=interval_ms,
        blit=False,
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Already returns IPython.display.HTML; do not wrap with HTML() again.
animate_training_compare(history, history_scheduler)


# Test


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "cifar10_best.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")

test loss 1.0605 | test acc 0.6259


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "cifar10_best_scheduler.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")

test loss 0.8046 | test acc 0.7327
